In [16]:
import os
from langchain_openrouter import ChatOpenRouter
from dotenv import load_dotenv
load_dotenv()

True

In [17]:
from langchain_openrouter import ChatOpenRouter

llm = ChatOpenRouter(
    model="openai/gpt-4o-mini",
    temperature=0
)

response = llm.invoke("Hi")
print(response.content)

Hello! How can I assist you today?


## Utils

In [18]:
## DocLoader
from langchain_community.document_loaders import PyMuPDFLoader
def resume_loader(file_path):
    loader = PyMuPDFLoader(file_path)
    doc_list =  loader.load()
    return '\n'.join([i.page_content for i in doc_list if i.page_content])

def text_loader(file_path):
    with open(file_path, 'r') as f:
        return f.read()

## Analyzer

In [19]:
### Resume Analyzer
resume_text = resume_loader("/Users/munna/Projects/QBS/careerpilot_demo/docs/Mahmud_Hasan_Munna_BL.pdf")
jd_text = text_loader("/Users/munna/Projects/QBS/careerpilot_demo/docs/jd.txt")

In [20]:
### Promt --> LLM --> Structured Output Response
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import HumanMessage,SystemMessage
from pydantic import BaseModel, Field
from typing import List

In [21]:
def analyze_resume(resume_text, jd_text):
    class ResumeAnalysis(BaseModel):
        strong_points: List[str] = Field(description="List of strong points in the resume")
        weak_points: List[str] = Field(description="List of weak points in the resume")

    analyze_llm = llm.with_structured_output(ResumeAnalysis)
    prompt = ChatPromptTemplate.from_messages(
        [
            (
                "system",
                """
                You are an expert AI assistant specializing in Resume Strong and Weakness Analysis.

                Responsibilities:
                - You will be provided with a resume and a job description.
                - Analyze the resume in the context of the job description.
                - You will provide a structured analysis highlighting the strong points and weak points of the resume with respect to the job description.
                - Max allowed strong points: 5
                - Max allowed weak points: 5

                """
            ),
            (
                "human",
                """
                Resume:
                {resume}

                JD:
                {jd}
                """
            ),
        ]
    )
    analyze_chain = prompt | analyze_llm
    result = analyze_chain.invoke(
        {
            "resume": resume_text,
            "jd": jd_text,
        }
    )
    return dict(result)

In [22]:
analyze_resume(resume_text, jd_text)

{'strong_points': ["4+ years of experience in AI/ML engineering with a focus on deploying production-grade machine learning systems, aligning with the job's requirement for extensive experience in AI/ML engineering.",
  "Hands-on expertise in LLM/GenAI systems, including RAG architectures and prompt engineering, which directly relates to the job's focus on Generative AI platforms and applications.",
  'Experience with MLOps, CI/CD for ML, and cloud platforms (AWS), which is essential for architecting scalable and cost-efficient AI infrastructure as mentioned in the job description.',
  'Proven track record in automating processes and improving efficiency, such as reducing onboarding time by 90% and saving 80 hours/month, demonstrating the ability to drive impactful AI solutions.',
  'Strong technical skills in Python, FastAPI, Docker, and various ML frameworks, which are crucial for backend engineering and operationalizing production-scale GenAI applications.'],
 'weak_points': ['Only 

## Resume Feedback

In [23]:
def get_resume_feedback(resume_text, jd_text):
    class ResumeFeedback(BaseModel):
        to_be_added: List[str] = Field(description="Elements that are missing in the resume but are relevant to the job description")
        to_be_deleted: List[str] = Field(description="Elements that are present in the resume but are not relevant to the job description")

    feedback_llm = llm.with_structured_output(ResumeFeedback)
    prompt = ChatPromptTemplate.from_messages(
        [
            (
                "system",
                """
                You are an expert AI assistant specializing in Resume Providing Resume Feedback.

                Responsibilities:
                - You will be provided with a resume and a job description.
                - Analyze the resume in the context of the job description.
                - You will provide a structured feedback highlighting the elements that are missing in the resume but are relevant to the job description (to_be_added) and the elements that are present in the resume but are not relevant to the job description (to_be_deleted).


                """
            ),
            (
                "human",
                """
                Resume:
                {resume}

                JD:
                {jd}
                """
            ),
        ]
    )
    feedback_chain = prompt | feedback_llm
    result = feedback_chain.invoke(
        {
            "resume": resume_text,
            "jd": jd_text,
        }
    )
    return dict(result)

In [24]:
get_resume_feedback(resume_text, jd_text)

{'to_be_added': ['Experience leading technical teams or AI engineering squads',
  'Expertise in architecting distributed systems and operationalizing production-scale GenAI applications',
  'Hands-on experience with LLM application development, prompt engineering frameworks, RAG architectures, AI agents and tool-calling systems',
  'Experience with AI infrastructure including GPU-based inference systems, open-source LLM deployment, fine-tuning techniques (LoRA/PEFT)',
  'Experience with AI governance, responsible AI, and enterprise compliance frameworks',
  'Strong stakeholder influence, business communication, and presentation skills',
  'Contributions to open-source projects, patents, conferences, or published research'],
 'to_be_deleted': ['Experience in telecom-domain processing 1B+ records (not relevant to GenAI)',
  'Specific mention of telecom recharge forecasting (not relevant to the broader GenAI focus)',
  'Details on financial and operational reports (not directly related to